<a href="https://colab.research.google.com/github/ShampooingBon/Gemibon/blob/main/Gemibon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# 1. PARAMÈTRES ET CONFIGURATION DE BASE
TAILLE_VOCABULAIRE = 256  # Encodage ASCII basique pour démarrer proprement sans gros tokenizer externe
DIM_EMBEDDING = 64        # Dimension des vecteurs pour les mots
NB_TETE_ATTENTION = 4     # Nombre de têtes pour le mécanisme d'auto-attention
NB_COUCHES = 2            # Nombre de blocs de transformeurs empilés
LONGUEUR_SEQUENCE = 32    # Taille de la fenêtre de contexte
TAILLE_LOT = 16
NB_EPOQUES = 50

# Ton texte d'entraînement initial (avec tes règles et tes entreprises)
texte_initial = """
Bonjour, je suis Gemibon, l'assistant IA officiel de Shampooing Bon et de l'entreprise BV.
Shampooing Bon propose des produits de soins de haute qualite.
Entreprise BV gere le developpement web, l'innovation et les projets technologiques.
Les utilisateurs doivent respecter la communaute et parler avec politesse.
Les insultes et les gros mots sont strictement interdits sur nos plateformes.
Seul le createur possede le code secret G-18sb pour debroquer un utilisateur banni.
"""

# Transformation du texte en indices numériques (tokens)
donnees_bytes = texte_initial.encode('utf-8')
donnees_trous = torch.tensor([b for b in donnees_bytes], dtype=torch.long)

def obtenir_lot():
    # Génération de mini-lots aléatoires pour l'entraînement
    idx = torch.randint(0, len(donnees_trous) - LONGUEUR_SEQUENCE, (TAILLE_LOT,))
    x = torch.stack([donnees_trous[i:i+LONGUEUR_SEQUENCE] for i in idx])
    y = torch.stack([donnees_trous[i+1:i+LONGUEUR_SEQUENCE+1] for i in idx])
    return x, y

# 2. ARCHITECTURE DE TYPE "TRANSFORMER" SIMPLIFIÉE
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(TAILLE_VOCABULAIRE, DIM_EMBEDDING)
        self.position_embedding = nn.Embedding(LONGUEUR_SEQUENCE, DIM_EMBEDDING)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=DIM_EMBEDDING,
            nhead=NB_TETE_ATTENTION,
            dim_feedforward=128,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=NB_COUCHES)
        self.fc_out = nn.Linear(DIM_EMBEDDING, TAILLE_VOCABULAIRE)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)

        tok_emb = self.embedding(idx)
        pos_emb = self.position_embedding(pos)
        x = tok_emb + pos_emb

        # Masque causal pour éviter que le modèle ne "triche" en regardant le futur
        masque = nn.Transformer.generate_square_subsequent_mask(T).to(idx.device)

        x = self.transformer_encoder(x, mask=masque, is_causal=True)
        logits = self.fc_out(x)
        return logits

# 3. INITIALISATION ET ENTRAÎNEMENT
modele = MiniTransformer()
optimiseur = optim.AdamW(modele.parameters(), lr=1e-3)
critere = nn.CrossEntropyLoss()

print("--- DEBUT DE L'ENTRAÎNEMENT DU MINI-TRANSFORMER ---")
modele.train()
for epoq in range(NB_EPOQUES):
    xb, yb = obtenir_lot()
    optimiseur.zero_grad()
    logits = modele(xb)

    # Réajustement des dimensions pour calculer la perte (loss)
    perte = critere(logits.view(-1, TAILLE_VOCABULAIRE), yb.view(-1))
    perte.backward()
    optimiseur.step()

    if (epoq + 1) % 10 == 0:
        print(f"Époque {epoq+1}/{NB_EPOQUES} - Perte : {perte.item():.4f}")

# 4. SAUVEGARDE DU MODÈLE
nom_fichier = "modele_gemibon_transformer.pt"
torch.save(modele.state_dict(), nom_fichier)
print(f"=== ENTRAÎNEMENT TERMINE : Modèle sauvegardé sous '{nom_fichier}' ===")

--- DEBUT DE L'ENTRAÎNEMENT DU MINI-TRANSFORMER ---
Époque 10/50 - Perte : 4.5650
Époque 20/50 - Perte : 3.7380
Époque 30/50 - Perte : 3.2588
Époque 40/50 - Perte : 3.0238
Époque 50/50 - Perte : 2.8223
=== ENTRAÎNEMENT TERMINE : Modèle sauvegardé sous 'modele_gemibon_transformer.pt' ===
